# Check object matching results

Examines per-catalog coverage, SNR distributions, and spatial layouts of the multi-catalog object match.

In [ ]:
# Standard imports
import matplotlib.pyplot as plt
import numpy as np
import tables_io
import hpmcm

In [ ]:
tract = 8006

# Input catalog files and IDs for the object meta-match
all_obj_stats = tables_io.read(f'{tract}/all_match_object_stats.pq')
all_obj_assoc = tables_io.read(f'{tract}/all_match_object_assoc.pq')
object_in = tables_io.read(f'{tract}/object_reduced_{tract}.parq')
anacal_in = tables_io.read(f'{tract}/anacal_reduced_{tract}.parq')
desc_md_in = tables_io.read(f'{tract}/desc_md_match_object_stats.pq')
rubin_md_in = tables_io.read(f'{tract}/rubin_md_match_object_stats.pq')

all_files = [
    f'{tract}/object_reduced_{tract}.parq',
    f'{tract}/anacal_reduced_{tract}.parq',
    f'{tract}/desc_md_match_object_stats.pq',
    f'{tract}/rubin_md_match_object_stats.pq',
]
all_ids = list(range(len(all_files)))

In [ ]:
desc_md_in.object_id

In [ ]:
# Build wide joined table
joined_obj = hpmcm.output_tables.buildJoinedObjectTable(
    all_obj_stats,
    all_obj_assoc,
    all_files,
    all_ids,
)

In [ ]:
# SNR distributions split by which catalogs detected each object
bins = np.logspace(0, 5, 51)
for i in range(4):
    mask_bit = 1 << i
    mask = np.bitwise_and(joined_obj.catalog_mask, mask_bit) == mask_bit
    _ = plt.hist(joined_obj[mask].snr, bins=bins, alpha=0.1)
_ = plt.xscale('log')
_ = plt.yscale('log')

In [ ]:
# 2D histogram: which catalogs detected each object vs SNR
_ = plt.hist2d(joined_obj.catalog_mask, joined_obj.snr, bins=(np.linspace(0.5, 15.5, 16), np.logspace(0, 5, 51)), norm='log')
_ = plt.yscale('log')

In [ ]:
_ = plt.hist(joined_obj.catalog_mask, bins=np.linspace(0.5, 15.5, 16))
_ = plt.yscale('log')

In [ ]:
cut = np.bitwise_and(joined_obj.n_sib_objects==1, joined_obj.n_sib_srcs>4)
print(cut.sum())

In [ ]:
good_mask = joined_obj.catalog_mask == 15
bad_mask = joined_obj.catalog_mask != 15
good_sub = joined_obj[good_mask]
bad_sub = joined_obj[bad_mask]

In [ ]:
_ = plt.scatter(bad_sub.ra[::5], bad_sub.dec[::5], s=1)
_ = plt.scatter(good_sub.ra[::5], good_sub.dec[::5], s=1)

In [ ]:
_ = plt.scatter(bad_sub.x_cent[::5], bad_sub.y_cent[::5], s=1)
_ = plt.scatter(good_sub.x_cent[::5], good_sub.y_cent[::5], s=1)

In [ ]:
_ = plt.hist2d(3600*(good_sub.ra-good_sub.ra_0), 3600*(good_sub.dec-good_sub.dec_0),
               bins=(np.linspace(-0.8, 0.8, 81), np.linspace(-0.8, 0.8, 81)), norm='log')

In [ ]:
_ = plt.hist(bad_sub.x_cent, bins=np.linspace(-5, 155, 161))
_ = plt.hist(good_sub.x_cent, bins=np.linspace(-5, 155, 161))